# VisionGuard AI — Defect Localization

This notebook performs detailed defect localization for the five selected MVTec AD categories using the trained category-specific CNN autoencoders.

The workflow focuses on turning reconstruction differences into spatial anomaly maps, binary defect masks, connected regions, bounding boxes, overlays, and pixel-level evaluation.

The notebook is deliberately split into many small execution cells so that each stage can be inspected and debugged independently.

## Localization workflow

The notebook:

- discovers the prepared MVTec dataset and trained category models automatically
- preserves the same 128×128 RGB inference contract used by the training workflow
- reconstructs test images with category-specific autoencoders
- creates per-pixel reconstruction-error maps
- calibrates category-specific pixel thresholds from normal validation images
- applies thresholding and morphological cleanup
- extracts connected defect regions and bounding boxes
- matches defective test images to their actual ground-truth masks
- calculates IoU, Dice, precision, recall, pixel accuracy, and area statistics
- creates detailed qualitative localization visualizations
- exports every Matplotlib and Plotly visualization to the `images` artifact folder using title-derived filenames
- avoids displaying filesystem paths in notebook outputs

## Evaluation principle

For an input image `x` and its autoencoder reconstruction `x̂`, the channel-wise absolute reconstruction discrepancy is converted into a spatial error map.

A pixel is considered suspicious when its error exceeds a category-specific threshold estimated from normal validation images.

The primary localization comparison is then:

**predicted anomaly mask vs. ground-truth defect mask**

The notebook treats the ground-truth directory structure dynamically and never invents defect names.

## Install dependencies

In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
    "plotly": "plotly",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow",
    "cv2": "opencv-python",
    "kaleido": "kaleido",
}

missing_packages = [
    package
    for module_name, package in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
        )
    except Exception as exc:
        raise RuntimeError(
            "Dependency installation failed. Resolve the package issue and rerun this notebook."
        ) from exc

print("Dependencies ready.")

Dependencies ready.


## Imports

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import re
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

import cv2

import tensorflow as tf
from tensorflow import keras

ImageFile.LOAD_TRUNCATED_IMAGES = False
warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Plotly:", __import__("plotly").__version__)

TensorFlow: 2.20.0
NumPy: 2.5.3
Pandas: 3.0.6
Plotly: 7.1.0


## Configuration

In [3]:
CATEGORIES = [
    "screw",
    "bottle",
    "capsule",
    "metal_nut",
    "hazelnut",
]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128
IMAGE_CHANNELS = 3

VALIDATION_SIZE = 0.10
SEED = 42

IMAGE_THRESHOLD_PERCENTILE = 99.0
PIXEL_THRESHOLD_PERCENTILE = 99.5

MORPH_KERNEL_SIZE = 3
MIN_COMPONENT_AREA = 12

PLOT_WIDTH = 1400
PLOT_HEIGHT = 850
PLOT_SCALE = 2

IMAGE_DIR = Path("images")
TABLE_DIR = Path("table")

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("Configuration initialized.")
print("Categories:", len(CATEGORIES))
print("Inference resolution:", f"{IMAGE_WIDTH} × {IMAGE_HEIGHT}")
print("Pixel threshold percentile:", PIXEL_THRESHOLD_PERCENTILE)

Configuration initialized.
Categories: 5
Inference resolution: 128 × 128
Pixel threshold percentile: 99.5


## TensorFlow runtime

In [4]:
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

gpu_devices = tf.config.list_physical_devices("GPU")

print("GPU devices detected:", len(gpu_devices))

for gpu in gpu_devices:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

GPU devices detected: 1


## Visualization filename utilities

In [5]:
def sanitize_title_filename(title):
    text = str(title).strip()
    text = re.sub(r"[^\w\s\-\.]", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    text = text.strip("_.")
    return text or "visualization"


def image_output_file(title):
    return IMAGE_DIR / f"{sanitize_title_filename(title)}.png"


def html_output_file(title):
    return IMAGE_DIR / f"{sanitize_title_filename(title)}.html"

print("Title-based filename utilities ready.")

Title-based filename utilities ready.


## Matplotlib export helper

In [6]:
def save_matplotlib_figure(fig, title, dpi=220, close=True):
    output = image_output_file(title)

    fig.savefig(
        output,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )

    if close:
        plt.close(fig)

    return output.name

## Plotly export helper

In [7]:
def save_plotly_figure(fig, title, width=PLOT_WIDTH, height=PLOT_HEIGHT):
    html_output = html_output_file(title)
    png_output = image_output_file(title)

    fig.write_html(
        html_output,
        include_plotlyjs="cdn",
    )

    png_saved = False

    try:
        fig.write_image(
            png_output,
            width=width,
            height=height,
            scale=PLOT_SCALE,
        )
        png_saved = True
    except Exception:
        pass

    return {
        "HTML": html_output.name,
        "PNG": png_output.name if png_saved else None,
    }

## Display-safe table helper

In [8]:
PATH_COLUMNS = {
    "path",
    "filepath",
    "file_path",
    "image_path",
    "mask_path",
    "ground_truth_path",
    "model_path",
    "dataset_path",
}


def path_like_column(column_name):
    normalized = str(column_name).strip().lower().replace(" ", "_")
    return normalized in PATH_COLUMNS or normalized.endswith("_path")


def display_safe(df):
    safe = df.copy()

    drop_columns = [
        column
        for column in safe.columns
        if path_like_column(column)
        or str(column).startswith("_")
    ]

    if drop_columns:
        safe = safe.drop(columns=drop_columns)

    return safe


def save_table(df, name):
    clean = display_safe(df)
    clean.to_csv(TABLE_DIR / f"{sanitize_title_filename(name)}.csv", index=False)
    return clean

## Filesystem helpers

In [9]:
def is_image_file(path):
    path = Path(path)
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def image_files(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    return sorted(
        path
        for path in folder.rglob("*")
        if is_image_file(path)
    )


def find_named_directories(name):
    base = Path.cwd()
    candidates = []

    direct = base / name
    if direct.is_dir():
        candidates.append(direct)

    excluded = {
        ".git",
        "__pycache__",
        ".ipynb_checkpoints",
        ".cache",
        "node_modules",
    }

    try:
        for candidate in base.rglob(name):
            if (
                candidate.is_dir()
                and not any(part in excluded for part in candidate.parts)
            ):
                if candidate not in candidates:
                    candidates.append(candidate)
    except Exception:
        pass

    return sorted(candidates)

## Discover prepared dataset

In [10]:
def find_dataset_root():
    existing = globals().get("DATASET_ROOT")

    if existing:
        candidate = Path(existing)
        if candidate.is_dir():
            if any((candidate / category / "test").is_dir() for category in CATEGORIES):
                return candidate

    candidates = find_named_directories("visionguard_data")

    for candidate in candidates:
        if any((candidate / category / "test").is_dir() for category in CATEGORIES):
            return candidate

    raise FileNotFoundError(
        "The prepared VisionGuard dataset could not be discovered. "
        "Run the dataset preparation notebook first."
    )


DATA_ROOT = find_dataset_root()

print("Prepared dataset discovered.")

Prepared dataset discovered.


## Validate dataset categories

In [11]:
dataset_rows = []

for category in CATEGORIES:
    category_root = DATA_ROOT / category

    dataset_rows.append({
        "Category": category,
        "Available": category_root.is_dir(),
        "Train": (category_root / "train").is_dir(),
        "Test": (category_root / "test").is_dir(),
        "Ground Truth": (category_root / "ground_truth").is_dir(),
    })

dataset_status_df = pd.DataFrame(dataset_rows)

display(dataset_status_df)

available_categories = [
    row["Category"]
    for _, row in dataset_status_df.iterrows()
    if row["Test"]
]

if not available_categories:
    raise RuntimeError("No requested test categories are available.")

save_table(
    dataset_status_df,
    "VisionGuard AI Dataset Category Availability",
)

,Category,Available,Train,Test,Ground Truth
0,screw,True,True,True,True
1,bottle,True,True,True,True
2,capsule,True,True,True,True
3,metal_nut,True,True,True,True
4,hazelnut,True,True,True,True


,Category,Available,Train,Test,Ground Truth
0,screw,True,True,True,True
1,bottle,True,True,True,True
2,capsule,True,True,True,True
3,metal_nut,True,True,True,True
4,hazelnut,True,True,True,True


## Discover trained models

In [12]:
def model_candidates(category):
    candidates = []

    existing_registry = globals().get("best_models")

    if isinstance(existing_registry, dict) and category in existing_registry:
        candidate = existing_registry[category]
        if isinstance(candidate, keras.Model):
            return candidate

        try:
            candidate = Path(candidate)
            if candidate.is_file():
                candidates.append(candidate)
        except Exception:
            pass

    preferred = Path("models") / category / "best_model.keras"

    if preferred.is_file():
        candidates.append(preferred)

    try:
        for candidate in Path.cwd().rglob("best_model.keras"):
            if candidate.is_file() and candidate.parent.name == category:
                candidates.append(candidate)
    except Exception:
        pass

    unique = sorted(set(candidates))

    if not unique:
        return None

    return unique[0]

## Load category-specific models

In [13]:
model_registry = {}
model_rows = []

for category in CATEGORIES:
    candidate = model_candidates(category)

    if isinstance(candidate, keras.Model):
        model_registry[category] = candidate
        model_rows.append({
            "Category": category,
            "Model Available": True,
            "Load Status": "Existing model object",
        })
        continue

    if candidate is None:
        model_rows.append({
            "Category": category,
            "Model Available": False,
            "Load Status": "Model not found",
        })
        continue

    try:
        model_registry[category] = keras.models.load_model(candidate, compile=False)
        model_rows.append({
            "Category": category,
            "Model Available": True,
            "Load Status": "Loaded successfully",
        })
    except Exception as exc:
        model_rows.append({
            "Category": category,
            "Model Available": False,
            "Load Status": f"Load failed: {type(exc).__name__}",
        })

model_status_df = pd.DataFrame(model_rows)
display(model_status_df)

ready_categories = [
    category
    for category in CATEGORIES
    if category in model_registry
]

if not ready_categories:
    raise RuntimeError("No trained models are available for localization.")

save_table(
    model_status_df,
    "VisionGuard AI Model Availability",
)

I0000 00:00:1790961027.114657    2426 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


,Category,Model Available,Load Status
0,screw,True,Loaded successfully
1,bottle,True,Loaded successfully
2,capsule,True,Loaded successfully
3,metal_nut,True,Loaded successfully
4,hazelnut,True,Loaded successfully


,Category,Model Available,Load Status
0,screw,True,Loaded successfully
1,bottle,True,Loaded successfully
2,capsule,True,Loaded successfully
3,metal_nut,True,Loaded successfully
4,hazelnut,True,Loaded successfully


## Inspect model input and output contract

In [14]:
contract_rows = []

for category, model in model_registry.items():
    input_shape = tuple(model.input_shape)
    output_shape = tuple(model.output_shape)

    contract_rows.append({
        "Category": category,
        "Input Shape": str(input_shape),
        "Output Shape": str(output_shape),
        "Parameters": int(model.count_params()),
        "Shape Valid": (
            input_shape[-3:] == (
                IMAGE_HEIGHT,
                IMAGE_WIDTH,
                IMAGE_CHANNELS,
            )
            and output_shape[-3:] == (
                IMAGE_HEIGHT,
                IMAGE_WIDTH,
                IMAGE_CHANNELS,
            )
        ),
    })

model_contract_df = pd.DataFrame(contract_rows)

display(model_contract_df)

if not model_contract_df["Shape Valid"].all():
    raise ValueError(
        "At least one available model does not match the expected image tensor shape."
    )

save_table(
    model_contract_df,
    "VisionGuard AI Model Input Output Contract",
)

,Category,Input Shape,Output Shape,Parameters,Shape Valid
0,screw,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
1,bottle,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
2,capsule,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
3,metal_nut,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
4,hazelnut,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True


,Category,Input Shape,Output Shape,Parameters,Shape Valid
0,screw,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
1,bottle,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
2,capsule,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
3,metal_nut,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True
4,hazelnut,"(None, 128, 128, 3)","(None, 128, 128, 3)",778371,True


## Model parameter comparison

In [15]:
fig = px.bar(
    model_contract_df,
    x="Category",
    y="Parameters",
    text="Parameters",
    title="VisionGuard AI — Autoencoder Parameter Count by Category",
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Trainable Parameter Count",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Autoencoder Parameter Count by Category",
)

fig.show()

## Image loading utilities

In [16]:
def load_rgb_array(image_path):
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image = image.resize(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.Resampling.LANCZOS,
        )

        array = np.asarray(
            image,
            dtype=np.float32,
        ) / 255.0

    return array


def load_mask_array(mask_path):
    with Image.open(mask_path) as mask:
        mask = mask.convert("L")
        mask = mask.resize(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.Resampling.NEAREST,
        )

        array = np.asarray(
            mask,
            dtype=np.uint8,
        )

    return array > 0

## Reconstruction error utilities

In [17]:
def reconstruction_metrics(original, reconstruction):
    original = np.asarray(original, dtype=np.float32)
    reconstruction = np.asarray(reconstruction, dtype=np.float32)

    difference = original - reconstruction
    absolute = np.abs(difference)
    squared = np.square(difference)

    pixel_mae = np.mean(absolute, axis=-1)
    pixel_mse = np.mean(squared, axis=-1)

    return {
        "mse": float(np.mean(squared)),
        "mae": float(np.mean(absolute)),
        "maximum_error": float(np.max(absolute)),
        "pixel_mae": pixel_mae,
        "pixel_mse": pixel_mse,
    }

## Single-image reconstruction

In [18]:
def reconstruct_image(model, image_path):
    image = load_rgb_array(image_path)

    reconstruction = model.predict(
        image[None, ...],
        verbose=0,
    )[0]

    reconstruction = np.clip(
        reconstruction,
        0.0,
        1.0,
    )

    metrics = reconstruction_metrics(
        image,
        reconstruction,
    )

    return image, reconstruction, metrics

## Ground-truth search strategy

In [19]:
def normalize_stem(text):
    value = Path(text).stem.lower()

    if value.endswith("_mask"):
        value = value[:-5]

    return value


def discover_ground_truth_index(category):
    index = {}

    root = DATA_ROOT / category / "ground_truth"

    for mask_path in image_files(root):
        key = normalize_stem(mask_path.name)

        index.setdefault(
            key,
            []
        ).append(mask_path)

    return index

## Build ground-truth index

In [20]:
ground_truth_registry = {
    category: discover_ground_truth_index(category)
    for category in ready_categories
}

ground_truth_count_rows = []

for category in ready_categories:
    ground_truth_count_rows.append({
        "Category": category,
        "Ground Truth Images": sum(
            len(items)
            for items in ground_truth_registry[category].values()
        ),
        "Unique Sample Keys": len(
            ground_truth_registry[category]
        ),
    })

ground_truth_index_df = pd.DataFrame(
    ground_truth_count_rows
)

display(ground_truth_index_df)

save_table(
    ground_truth_index_df,
    "VisionGuard AI Ground Truth Index Summary",
)

,Category,Ground Truth Images,Unique Sample Keys
0,screw,119,25
1,bottle,63,22
2,capsule,109,23
3,metal_nut,93,25
4,hazelnut,70,18


,Category,Ground Truth Images,Unique Sample Keys
0,screw,119,25
1,bottle,63,22
2,capsule,109,23
3,metal_nut,93,25
4,hazelnut,70,18


## Match a test image with its ground-truth mask

In [21]:
def find_ground_truth_mask(category, defect_type, image_path):
    if str(defect_type).lower() == "good":
        return None

    index = ground_truth_registry.get(category, {})

    stem = normalize_stem(Path(image_path).name)

    direct_candidates = index.get(stem, [])

    if direct_candidates:
        defect_candidates = [
            candidate
            for candidate in direct_candidates
            if defect_type.lower() in str(candidate.parent.name).lower()
        ]

        if defect_candidates:
            return sorted(defect_candidates)[0]

        return sorted(direct_candidates)[0]

    target_fragment = str(defect_type).lower()

    for key, candidates in index.items():
        for candidate in candidates:
            if (
                key == stem
                and target_fragment in str(candidate.parent.name).lower()
            ):
                return candidate

    return None

## Discover actual test defect types

In [22]:
def get_test_defect_types(category):
    root = DATA_ROOT / category / "test"

    if not root.is_dir():
        return []

    return sorted(
        folder.name
        for folder in root.iterdir()
        if folder.is_dir()
    )


test_defect_inventory = []

for category in ready_categories:
    for defect_type in get_test_defect_types(category):
        test_defect_inventory.append({
            "Category": category,
            "Defect Type": defect_type,
            "Images": len(
                image_files(
                    DATA_ROOT
                    / category
                    / "test"
                    / defect_type
                )
            ),
        })

test_defect_df = pd.DataFrame(
    test_defect_inventory
)

display(test_defect_df)

save_table(
    test_defect_df,
    "VisionGuard AI Test Defect Inventory",
)

,Category,Defect Type,Images
0,screw,good,41
1,screw,manipulated_front,24
2,screw,scratch_head,24
3,screw,scratch_neck,25
4,screw,thread_side,23
5,screw,thread_top,23
6,bottle,broken_large,20
7,bottle,broken_small,22
8,bottle,contamination,21
9,bottle,good,20


,Category,Defect Type,Images
0,screw,good,41
1,screw,manipulated_front,24
2,screw,scratch_head,24
3,screw,scratch_neck,25
4,screw,thread_side,23
5,screw,thread_top,23
6,bottle,broken_large,20
7,bottle,broken_small,22
8,bottle,contamination,21
9,bottle,good,20


## Test defect distribution

In [23]:
fig = px.bar(
    test_defect_df,
    x="Defect Type",
    y="Images",
    color="Category",
    barmode="group",
    text_auto=True,
    title="VisionGuard AI — Test Images by Actual Defect Type",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Defect Type",
    yaxis_title="Number of Images",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Test Images by Actual Defect Type",
)

fig.show()

## Build test manifest

In [24]:
manifest_rows = []

for category in ready_categories:
    test_root = DATA_ROOT / category / "test"

    for defect_type in get_test_defect_types(category):
        defect_root = test_root / defect_type

        for image_path in image_files(defect_root):
            mask_path = find_ground_truth_mask(
                category,
                defect_type,
                image_path,
            )

            manifest_rows.append({
                "Category": category,
                "Defect Type": defect_type,
                "Label": 0 if defect_type.lower() == "good" else 1,
                "Filename": image_path.name,
                "Ground Truth Available": mask_path is not None,
                "_image_path": image_path,
                "_mask_path": mask_path,
            })

test_manifest_df = pd.DataFrame(
    manifest_rows
)

if test_manifest_df.empty:
    raise RuntimeError("No test images were discovered.")

print("Test images discovered:", len(test_manifest_df))

Test images discovered: 600


## Test manifest summary

In [25]:
manifest_summary_df = (
    test_manifest_df
    .groupby(
        ["Category", "Defect Type", "Label"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Images"})
)

display(
    manifest_summary_df
)

save_table(
    manifest_summary_df,
    "VisionGuard AI Test Manifest Summary",
)

,Category,Defect Type,Label,Images
0,bottle,broken_large,1,20
1,bottle,broken_small,1,22
2,bottle,contamination,1,21
3,bottle,good,0,20
4,capsule,crack,1,23
5,capsule,faulty_imprint,1,22
6,capsule,good,0,23
7,capsule,poke,1,21
8,capsule,scratch,1,23
9,capsule,squeeze,1,20


,Category,Defect Type,Label,Images
0,bottle,broken_large,1,20
1,bottle,broken_small,1,22
2,bottle,contamination,1,21
3,bottle,good,0,20
4,capsule,crack,1,23
5,capsule,faulty_imprint,1,22
6,capsule,good,0,23
7,capsule,poke,1,21
8,capsule,scratch,1,23
9,capsule,squeeze,1,20


## Normal versus defective test balance

In [26]:
test_balance_df = (
    test_manifest_df
    .assign(
        Class=lambda df: np.where(
            df["Label"].eq(0),
            "Normal",
            "Defective",
        )
    )
    .groupby(
        ["Category", "Class"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Images"})
)

display(
    test_balance_df
)

save_table(
    test_balance_df,
    "VisionGuard AI Normal Versus Defective Test Balance",
)

,Category,Class,Images
0,bottle,Defective,63
1,bottle,Normal,20
2,capsule,Defective,109
3,capsule,Normal,23
4,hazelnut,Defective,70
5,hazelnut,Normal,40
6,metal_nut,Defective,93
7,metal_nut,Normal,22
8,screw,Defective,119
9,screw,Normal,41


,Category,Class,Images
0,bottle,Defective,63
1,bottle,Normal,20
2,capsule,Defective,109
3,capsule,Normal,23
4,hazelnut,Defective,70
5,hazelnut,Normal,40
6,metal_nut,Defective,93
7,metal_nut,Normal,22
8,screw,Defective,119
9,screw,Normal,41


## Plot normal versus defective balance

In [27]:
fig = px.bar(
    test_balance_df,
    x="Category",
    y="Images",
    color="Class",
    barmode="group",
    text="Images",
    title="VisionGuard AI — Normal Versus Defective Test Images",
    template="plotly_white",
)

fig.update_traces(
    textposition="outside",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Normal Versus Defective Test Images",
)

fig.show()

## Ground-truth availability audit

In [28]:
ground_truth_audit_df = (
    test_manifest_df
    .groupby("Category", as_index=False)
    .agg(
        Defective_Images=("Label", "sum"),
        Ground_Truth_Matched=("Ground Truth Available", "sum"),
    )
)

ground_truth_audit_df["Match Rate"] = np.where(
    ground_truth_audit_df["Defective_Images"] > 0,
    ground_truth_audit_df["Ground_Truth_Matched"]
    / ground_truth_audit_df["Defective_Images"],
    np.nan,
)

display(
    ground_truth_audit_df
)

save_table(
    ground_truth_audit_df,
    "VisionGuard AI Ground Truth Availability Audit",
)

,Category,Defective_Images,Ground_Truth_Matched,Match Rate
0,bottle,63,63,1.0
1,capsule,109,109,1.0
2,hazelnut,70,70,1.0
3,metal_nut,93,93,1.0
4,screw,119,119,1.0


,Category,Defective_Images,Ground_Truth_Matched,Match Rate
0,bottle,63,63,1.0
1,capsule,109,109,1.0
2,hazelnut,70,70,1.0
3,metal_nut,93,93,1.0
4,screw,119,119,1.0


## Ground-truth match-rate visualization

In [29]:
fig = px.bar(
    ground_truth_audit_df,
    x="Category",
    y="Match Rate",
    text="Match Rate",
    title="VisionGuard AI — Ground Truth Match Rate for Defective Images",
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{y:.1%}",
    textposition="outside",
)

fig.update_layout(
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1],
    xaxis_title="Category",
    yaxis_title="Matched Ground Truth Rate",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Ground Truth Match Rate for Defective Images",
)

fig.show()

## Recreate normal validation split

In [30]:
def normal_training_images(category):
    return image_files(
        DATA_ROOT
        / category
        / "train"
        / "good"
    )


normal_validation_registry = {}
validation_split_rows = []

for category in ready_categories:
    normal_files = normal_training_images(category)

    if len(normal_files) < 2:
        continue

    train_files, validation_files = train_test_split(
        normal_files,
        test_size=VALIDATION_SIZE,
        random_state=SEED,
        shuffle=True,
    )

    normal_validation_registry[category] = sorted(
        validation_files
    )

    validation_split_rows.append({
        "Category": category,
        "Normal Training Images": len(train_files),
        "Normal Validation Images": len(validation_files),
    })

normal_validation_df = pd.DataFrame(
    validation_split_rows
)

display(normal_validation_df)

if normal_validation_df.empty:
    raise RuntimeError(
        "Normal validation images could not be created."
    )

,Category,Normal Training Images,Normal Validation Images
0,screw,288,32
1,bottle,188,21
2,capsule,197,22
3,metal_nut,198,22
4,hazelnut,351,40


## Validate normal split integrity

In [31]:
split_check_rows = []

for category in ready_categories:
    all_files = set(normal_training_images(category))
    validation_files = set(
        normal_validation_registry.get(category, [])
    )

    train_files = all_files - validation_files

    split_check_rows.append({
        "Category": category,
        "All Normal Images": len(all_files),
        "Validation Images": len(validation_files),
        "Remaining Training Images": len(train_files),
        "Disjoint": len(train_files.intersection(validation_files)) == 0,
    })

split_integrity_df = pd.DataFrame(split_check_rows)

display(split_integrity_df)

if not split_integrity_df["Disjoint"].all():
    raise ValueError("Normal training and validation sets overlap.")

,Category,All Normal Images,Validation Images,Remaining Training Images,Disjoint
0,screw,320,32,288,True
1,bottle,209,21,188,True
2,capsule,219,22,197,True
3,metal_nut,220,22,198,True
4,hazelnut,391,40,351,True


## Score one validation image

In [32]:
def validation_pixel_error(model, image_path):
    image, reconstruction, metrics = reconstruct_image(
        model,
        image_path,
    )

    return metrics["pixel_mae"]

## Estimate image-level reconstruction thresholds

In [33]:
image_threshold_rows = []
validation_error_registry = {}

for category in ready_categories:
    model = model_registry[category]
    validation_paths = normal_validation_registry.get(category, [])

    category_errors = []

    for image_path in validation_paths:
        _, _, metrics = reconstruct_image(
            model,
            image_path,
        )
        category_errors.append(metrics["mse"])

    validation_error_registry[category] = np.asarray(
        category_errors,
        dtype=np.float32,
    )

    if category_errors:
        threshold = float(
            np.percentile(
                category_errors,
                IMAGE_THRESHOLD_PERCENTILE,
            )
        )
    else:
        threshold = np.nan

    image_threshold_rows.append({
        "Category": category,
        "Validation Images": len(category_errors),
        "Image Threshold": threshold,
    })

image_threshold_df = pd.DataFrame(
    image_threshold_rows
)

display(
    image_threshold_df.round(8)
)

save_table(
    image_threshold_df,
    "VisionGuard AI Image Level Reconstruction Thresholds",
)

2026-10-02 17:10:33.099682: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-02 17:10:38.182721: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

,Category,Validation Images,Image Threshold
0,screw,32,0.024788
1,bottle,21,0.107541
2,capsule,22,0.066382
3,metal_nut,22,0.026352
4,hazelnut,40,0.008426


,Category,Validation Images,Image Threshold
0,screw,32,0.024788
1,bottle,21,0.107541
2,capsule,22,0.066382
3,metal_nut,22,0.026352
4,hazelnut,40,0.008426


## Estimate pixel-level reconstruction thresholds

In [34]:
pixel_threshold_rows = []
pixel_threshold_registry = {}

for category in ready_categories:
    model = model_registry[category]
    validation_paths = normal_validation_registry.get(category, [])

    pixel_values = []

    for image_path in validation_paths:
        pixel_map = validation_pixel_error(
            model,
            image_path,
        )

        pixel_values.append(
            pixel_map.reshape(-1)
        )

    if not pixel_values:
        continue

    pixel_values = np.concatenate(pixel_values)

    threshold = float(
        np.percentile(
            pixel_values,
            PIXEL_THRESHOLD_PERCENTILE,
        )
    )

    pixel_threshold_registry[category] = threshold

    pixel_threshold_rows.append({
        "Category": category,
        "Validation Pixels": len(pixel_values),
        "Pixel Threshold": threshold,
        "Percentile": PIXEL_THRESHOLD_PERCENTILE,
        "Validation Mean Error": float(np.mean(pixel_values)),
        "Validation Maximum Error": float(np.max(pixel_values)),
    })

pixel_threshold_df = pd.DataFrame(
    pixel_threshold_rows
)

display(
    pixel_threshold_df.round(8)
)

save_table(
    pixel_threshold_df,
    "VisionGuard AI Pixel Reconstruction Thresholds",
)

2026-10-02 17:10:58.607914: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-02 17:11:03.653869: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

,Category,Validation Pixels,Pixel Threshold,Percentile,Validation Mean Error,Validation Maximum Error
0,screw,524288,0.543348,99.5,0.095578,0.671358
1,bottle,344064,0.567017,99.5,0.306167,0.644802
2,capsule,360448,0.580867,99.5,0.190052,0.608774
3,metal_nut,360448,0.355388,99.5,0.119931,0.713884
4,hazelnut,655360,0.178612,99.5,0.056404,0.371989


,Category,Validation Pixels,Pixel Threshold,Percentile,Validation Mean Error,Validation Maximum Error
0,screw,524288,0.543348,99.5,0.095578,0.671358
1,bottle,344064,0.567017,99.5,0.306166,0.644802
2,capsule,360448,0.580867,99.5,0.190052,0.608774
3,metal_nut,360448,0.355388,99.5,0.119931,0.713884
4,hazelnut,655360,0.178612,99.5,0.056404,0.371989


## Pixel threshold visualization

In [35]:
fig = px.bar(
    pixel_threshold_df,
    x="Category",
    y="Pixel Threshold",
    text="Pixel Threshold",
    title="VisionGuard AI — Category-Specific Pixel Error Thresholds",
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{y:.6f}",
    textposition="outside",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Pixel MAE Threshold",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Category-Specific Pixel Error Thresholds",
)

fig.show()

## Pixel-level anomaly mask

In [36]:
def create_raw_anomaly_mask(error_map, threshold):
    return np.asarray(error_map) > float(threshold)


def clean_anomaly_mask(binary_mask):
    mask = np.asarray(binary_mask, dtype=np.uint8)

    kernel = np.ones(
        (MORPH_KERNEL_SIZE, MORPH_KERNEL_SIZE),
        dtype=np.uint8,
    )

    opened = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel,
    )

    closed = cv2.morphologyEx(
        opened,
        cv2.MORPH_CLOSE,
        kernel,
    )

    if MIN_COMPONENT_AREA <= 1:
        return closed.astype(bool)

    component_count, labels, stats, _ = cv2.connectedComponentsWithStats(
        closed,
        connectivity=8,
    )

    cleaned = np.zeros_like(closed)

    for label in range(1, component_count):
        area = int(stats[label, cv2.CC_STAT_AREA])

        if area >= MIN_COMPONENT_AREA:
            cleaned[labels == label] = 1

    return cleaned.astype(bool)

## Connected-component analysis

In [37]:
def extract_components(binary_mask):
    mask = np.asarray(binary_mask, dtype=np.uint8)

    count, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask,
        connectivity=8,
    )

    rows = []

    for label in range(1, count):
        x = int(stats[label, cv2.CC_STAT_LEFT])
        y = int(stats[label, cv2.CC_STAT_TOP])
        width = int(stats[label, cv2.CC_STAT_WIDTH])
        height = int(stats[label, cv2.CC_STAT_HEIGHT])
        area = int(stats[label, cv2.CC_STAT_AREA])

        cx, cy = centroids[label]

        rows.append({
            "Component": label,
            "Area": area,
            "X": x,
            "Y": y,
            "Width": width,
            "Height": height,
            "Center X": float(cx),
            "Center Y": float(cy),
        })

    if not rows:
        return pd.DataFrame(
            columns=[
                "Component",
                "Area",
                "X",
                "Y",
                "Width",
                "Height",
                "Center X",
                "Center Y",
            ]
        )

    return pd.DataFrame(rows).sort_values(
        "Area",
        ascending=False,
    ).reset_index(drop=True)

## Localization metrics

In [38]:
def binary_overlap_metrics(prediction, target):
    prediction = np.asarray(prediction, dtype=bool)
    target = np.asarray(target, dtype=bool)

    pred_flat = prediction.reshape(-1)
    target_flat = target.reshape(-1)

    tp = int(np.logical_and(pred_flat, target_flat).sum())
    fp = int(np.logical_and(pred_flat, ~target_flat).sum())
    fn = int(np.logical_and(~pred_flat, target_flat).sum())
    tn = int(np.logical_and(~pred_flat, ~target_flat).sum())

    union = tp + fp + fn

    iou = tp / union if union else 1.0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 1.0

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    accuracy = (tp + tn) / len(target_flat) if len(target_flat) else np.nan

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall": recall,
        "Pixel Accuracy": accuracy,
        "Predicted Coverage": float(prediction.mean()),
        "Ground Truth Coverage": float(target.mean()),
    }

## Score and localize one image

In [39]:
def localize_one(model, image_path, category):
    image, reconstruction, metrics = reconstruct_image(
        model,
        image_path,
    )

    threshold = pixel_threshold_registry[category]

    raw_mask = create_raw_anomaly_mask(
        metrics["pixel_mae"],
        threshold,
    )

    cleaned_mask = clean_anomaly_mask(
        raw_mask
    )

    components = extract_components(
        cleaned_mask
    )

    return {
        "image": image,
        "reconstruction": reconstruction,
        "pixel_mae": metrics["pixel_mae"],
        "pixel_mse": metrics["pixel_mse"],
        "mse": metrics["mse"],
        "mae": metrics["mae"],
        "maximum_error": metrics["maximum_error"],
        "pixel_threshold": threshold,
        "raw_mask": raw_mask,
        "cleaned_mask": cleaned_mask,
        "components": components,
    }

## Smoke test localization

In [40]:
smoke_rows = []

for category in ready_categories:
    sample = test_manifest_df[
        test_manifest_df["Category"].eq(category)
    ]

    if sample.empty:
        continue

    row = sample.iloc[0]

    result = localize_one(
        model_registry[category],
        row["_image_path"],
        category,
    )

    smoke_rows.append({
        "Category": category,
        "Filename": row["Filename"],
        "Reconstruction MSE": result["mse"],
        "Mean Pixel Error": float(np.mean(result["pixel_mae"])),
        "Anomaly Pixels": int(result["cleaned_mask"].sum()),
        "Components": int(len(result["components"])),
        "Output Shape Valid": (
            result["reconstruction"].shape
            == (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS)
        ),
    })

smoke_df = pd.DataFrame(smoke_rows)

display(
    smoke_df.round(8)
)

if not smoke_df["Output Shape Valid"].all():
    raise ValueError("Localization smoke test failed.")

2026-10-02 17:11:13.855221: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,Category,Filename,Reconstruction MSE,Mean Pixel Error,Anomaly Pixels,Components,Output Shape Valid
0,screw,000.png,0.026038,0.099936,0,0,True
1,bottle,000.png,0.103742,0.301997,0,0,True
2,capsule,000.png,0.061398,0.189966,12,1,True
3,metal_nut,000.png,0.015742,0.111193,0,0,True
4,hazelnut,000.png,0.007223,0.062658,167,6,True


## Run localization over the complete test set

In [41]:
localization_records = []

for index, row in test_manifest_df.iterrows():
    category = row["Category"]

    if category not in model_registry:
        continue

    result = localize_one(
        model_registry[category],
        row["_image_path"],
        category,
    )

    components = result["components"]

    localization_records.append({
        "Category": category,
        "Defect Type": row["Defect Type"],
        "Label": int(row["Label"]),
        "Filename": row["Filename"],
        "Ground Truth Available": bool(row["Ground Truth Available"]),
        "Reconstruction MSE": result["mse"],
        "Reconstruction MAE": result["mae"],
        "Maximum Error": result["maximum_error"],
        "Pixel Threshold": result["pixel_threshold"],
        "Anomaly Pixels": int(result["cleaned_mask"].sum()),
        "Anomaly Coverage": float(result["cleaned_mask"].mean()),
        "Components": int(len(components)),
        "Largest Component": (
            int(components.iloc[0]["Area"])
            if not components.empty
            else 0
        ),
        "_image_path": row["_image_path"],
        "_mask_path": row["_mask_path"],
        "_pixel_mae": result["pixel_mae"],
        "_raw_mask": result["raw_mask"],
        "_cleaned_mask": result["cleaned_mask"],
        "_reconstruction": result["reconstruction"],
        "_image": result["image"],
    })

localization_df = pd.DataFrame(
    localization_records
)

if localization_df.empty:
    raise RuntimeError("Localization results are empty.")

print("Images localized:", len(localization_df))

2026-10-02 17:11:18.984317: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-02 17:11:24.147361: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Images localized: 600


## Localization result quality audit

In [42]:
quality_audit_df = pd.DataFrame([{
    "Images Evaluated": len(localization_df),
    "Finite Reconstruction Errors": np.isfinite(
        localization_df["Reconstruction MSE"]
    ).all(),
    "Finite Pixel Thresholds": np.isfinite(
        localization_df["Pixel Threshold"]
    ).all(),
    "Valid Anomaly Coverage": localization_df["Anomaly Coverage"].between(0, 1).all(),
    "Nonnegative Component Counts": (
        localization_df["Components"] >= 0
    ).all(),
}])

display(quality_audit_df)

if not quality_audit_df.iloc[0].all():
    raise ValueError("Localization quality audit failed.")

,Images Evaluated,Finite Reconstruction Errors,Finite Pixel Thresholds,Valid Anomaly Coverage,Nonnegative Component Counts
0,600,True,True,True,True


## Localization statistics by category

In [43]:
category_localization_summary_df = (
    localization_df
    .groupby("Category", as_index=False)
    .agg(
        Images=("Filename", "size"),
        Mean_MSE=("Reconstruction MSE", "mean"),
        Median_MSE=("Reconstruction MSE", "median"),
        Mean_Anomaly_Coverage=("Anomaly Coverage", "mean"),
        Mean_Components=("Components", "mean"),
        Mean_Largest_Component=("Largest Component", "mean"),
    )
)

display(
    category_localization_summary_df.round(6)
)

save_table(
    category_localization_summary_df,
    "VisionGuard AI Category Localization Statistics",
)

,Category,Images,Mean_MSE,Median_MSE,Mean_Anomaly_Coverage,Mean_Components,Mean_Largest_Component
0,bottle,83,0.104586,0.105063,0.000034,0.036145,0.554217
1,capsule,132,0.059926,0.061056,0.000927,0.250000,12.727273
2,hazelnut,110,0.005097,0.004502,0.003387,1.154545,34.409091
3,metal_nut,115,0.016555,0.016209,0.000188,0.113043,3.086957
4,screw,160,0.022231,0.021609,0.000388,0.193750,5.493750


,Category,Images,Mean_MSE,Median_MSE,Mean_Anomaly_Coverage,Mean_Components,Mean_Largest_Component
0,bottle,83,0.104586,0.105063,0.000034,0.036145,0.554217
1,capsule,132,0.059926,0.061056,0.000927,0.250000,12.727273
2,hazelnut,110,0.005097,0.004502,0.003387,1.154545,34.409091
3,metal_nut,115,0.016555,0.016209,0.000188,0.113043,3.086957
4,screw,160,0.022231,0.021609,0.000388,0.193750,5.493750


## Normal versus defective reconstruction error

In [44]:
fig = px.box(
    localization_df.assign(
        Class=np.where(
            localization_df["Label"].eq(0),
            "Normal",
            "Defective",
        )
    ),
    x="Category",
    y="Reconstruction MSE",
    color="Class",
    points=False,
    title="VisionGuard AI — Reconstruction Error by Test Class",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Reconstruction MSE",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Reconstruction Error by Test Class",
)

fig.show()

## Anomaly coverage by category

In [45]:
fig = px.box(
    localization_df,
    x="Category",
    y="Anomaly Coverage",
    points=False,
    title="VisionGuard AI — Predicted Anomaly Coverage by Category",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Predicted Anomaly Pixel Coverage",
    yaxis_tickformat=".1%",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Predicted Anomaly Coverage by Category",
)

fig.show()

## Extract pixel-level evaluation records

In [46]:
pixel_metric_rows = []

for _, row in localization_df.iterrows():
    if row["Label"] != 1:
        continue

    if row["_mask_path"] is None:
        continue

    target_mask = load_mask_array(
        row["_mask_path"]
    )

    metrics = binary_overlap_metrics(
        row["_cleaned_mask"],
        target_mask,
    )

    pixel_metric_rows.append({
        "Category": row["Category"],
        "Defect Type": row["Defect Type"],
        "Filename": row["Filename"],
        **metrics,
    })

pixel_metrics_df = pd.DataFrame(
    pixel_metric_rows
)

if pixel_metrics_df.empty:
    raise RuntimeError(
        "No defective images with matched ground-truth masks were available."
    )

display(
    display_safe(pixel_metrics_df).round(6).head(20)
)

save_table(
    pixel_metrics_df,
    "VisionGuard AI Pixel Localization Metrics",
)

,Category,Defect Type,Filename,TP,FP,FN,TN,IoU,Dice,Precision,Recall,Pixel Accuracy,Predicted Coverage,Ground Truth Coverage
0,screw,manipulated_front,000.png,0,0,65,16319,0.0,0.0,0.0,0.0,0.996033,0.000000,0.003967
1,screw,manipulated_front,001.png,0,0,44,16340,0.0,0.0,0.0,0.0,0.997314,0.000000,0.002686
2,screw,manipulated_front,002.png,0,0,37,16347,0.0,0.0,0.0,0.0,0.997742,0.000000,0.002258
3,screw,manipulated_front,003.png,0,0,78,16306,0.0,0.0,0.0,0.0,0.995239,0.000000,0.004761
4,screw,manipulated_front,004.png,0,0,35,16349,0.0,0.0,0.0,0.0,0.997864,0.000000,0.002136
5,screw,manipulated_front,005.png,0,0,51,16333,0.0,0.0,0.0,0.0,0.996887,0.000000,0.003113
6,screw,manipulated_front,006.png,0,0,56,16328,0.0,0.0,0.0,0.0,0.996582,0.000000,0.003418
7,screw,manipulated_front,007.png,0,0,79,16305,0.0,0.0,0.0,0.0,0.995178,0.000000,0.004822
8,screw,manipulated_front,008.png,0,0,62,16322,0.0,0.0,0.0,0.0,0.996216,0.000000,0.003784
9,screw,manipulated_front,009.png,0,0,106,16278,0.0,0.0,0.0,0.0,0.993530,0.000000,0.006470


,Category,Defect Type,Filename,TP,FP,FN,TN,IoU,Dice,Precision,Recall,Pixel Accuracy,Predicted Coverage,Ground Truth Coverage
0,screw,manipulated_front,000.png,0,0,65,16319,0.000000,0.000000,0.000000,0.000000,0.996033,0.000000,0.003967
1,screw,manipulated_front,001.png,0,0,44,16340,0.000000,0.000000,0.000000,0.000000,0.997314,0.000000,0.002686
2,screw,manipulated_front,002.png,0,0,37,16347,0.000000,0.000000,0.000000,0.000000,0.997742,0.000000,0.002258
3,screw,manipulated_front,003.png,0,0,78,16306,0.000000,0.000000,0.000000,0.000000,0.995239,0.000000,0.004761
4,screw,manipulated_front,004.png,0,0,35,16349,0.000000,0.000000,0.000000,0.000000,0.997864,0.000000,0.002136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,hazelnut,print,012.png,122,2,206,16054,0.369697,0.539823,0.983871,0.371951,0.987305,0.007568,0.020020
450,hazelnut,print,013.png,50,0,269,16065,0.156740,0.271003,1.000000,0.156740,0.983582,0.003052,0.019470
451,hazelnut,print,014.png,53,1,301,16029,0.149296,0.259804,0.981481,0.149718,0.981567,0.003296,0.021606
452,hazelnut,print,015.png,0,0,250,16134,0.000000,0.000000,0.000000,0.000000,0.984741,0.000000,0.015259


## Pixel metric summary by category

In [47]:
pixel_category_summary_df = (
    pixel_metrics_df
    .groupby("Category", as_index=False)
    .agg(
        Images=("Filename", "size"),
        Mean_IoU=("IoU", "mean"),
        Median_IoU=("IoU", "median"),
        Mean_Dice=("Dice", "mean"),
        Mean_Precision=("Precision", "mean"),
        Mean_Recall=("Recall", "mean"),
        Mean_Pixel_Accuracy=("Pixel Accuracy", "mean"),
        Mean_Predicted_Coverage=("Predicted Coverage", "mean"),
        Mean_Ground_Truth_Coverage=("Ground Truth Coverage", "mean"),
    )
)

display(
    pixel_category_summary_df.round(6)
)

save_table(
    pixel_category_summary_df,
    "VisionGuard AI Pixel Metric Summary by Category",
)

,Category,Images,Mean_IoU,Median_IoU,Mean_Dice,Mean_Precision,Mean_Recall,Mean_Pixel_Accuracy,Mean_Predicted_Coverage,Mean_Ground_Truth_Coverage
0,bottle,63,0.000000,0.000000,0.000000,0.000000,0.000000,0.923748,0.000016,0.076237
1,capsule,109,0.000054,0.000000,0.000107,0.001529,0.000055,0.988726,0.000167,0.011112
2,hazelnut,70,0.126182,0.058905,0.191049,0.539739,0.131022,0.970215,0.005171,0.033772
3,metal_nut,93,0.000238,0.000000,0.000473,0.064516,0.000238,0.855223,0.000179,0.144826
4,screw,119,0.000000,0.000000,0.000000,0.000000,0.000000,0.996523,0.000107,0.003371


,Category,Images,Mean_IoU,Median_IoU,Mean_Dice,Mean_Precision,Mean_Recall,Mean_Pixel_Accuracy,Mean_Predicted_Coverage,Mean_Ground_Truth_Coverage
0,bottle,63,0.000000,0.000000,0.000000,0.000000,0.000000,0.923748,0.000016,0.076237
1,capsule,109,0.000054,0.000000,0.000107,0.001529,0.000055,0.988726,0.000167,0.011112
2,hazelnut,70,0.126182,0.058905,0.191049,0.539739,0.131022,0.970215,0.005171,0.033772
3,metal_nut,93,0.000238,0.000000,0.000473,0.064516,0.000238,0.855223,0.000179,0.144826
4,screw,119,0.000000,0.000000,0.000000,0.000000,0.000000,0.996523,0.000107,0.003371


## IoU comparison

In [48]:
plot_df = pixel_category_summary_df.melt(
    id_vars="Category",
    value_vars=["Mean_IoU", "Mean_Dice"],
    var_name="Metric",
    value_name="Score",
)

fig = px.bar(
    plot_df,
    x="Category",
    y="Score",
    color="Metric",
    barmode="group",
    text="Score",
    title="VisionGuard AI — Mean IoU and Dice by Category",
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{y:.3f}",
    textposition="outside",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Score",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Mean IoU and Dice by Category",
)

fig.show()

## Precision, recall, and pixel accuracy

In [49]:
plot_df = pixel_category_summary_df.melt(
    id_vars="Category",
    value_vars=[
        "Mean_Precision",
        "Mean_Recall",
        "Mean_Pixel_Accuracy",
    ],
    var_name="Metric",
    value_name="Score",
)

fig = px.bar(
    plot_df,
    x="Category",
    y="Score",
    color="Metric",
    barmode="group",
    title="VisionGuard AI — Pixel Localization Precision Recall and Accuracy",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Score",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Pixel Localization Precision Recall and Accuracy",
)

fig.show()

## IoU distribution

In [50]:
fig = px.box(
    pixel_metrics_df,
    x="Category",
    y="IoU",
    points="outliers",
    title="VisionGuard AI — Defect Localization IoU Distribution",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="IoU",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Defect Localization IoU Distribution",
)

fig.show()

## Dice distribution

In [51]:
fig = px.box(
    pixel_metrics_df,
    x="Category",
    y="Dice",
    points="outliers",
    title="VisionGuard AI — Defect Localization Dice Distribution",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Dice Score",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Defect Localization Dice Distribution",
)

fig.show()

## Defect-type localization summary

In [52]:
defect_type_summary_df = (
    pixel_metrics_df
    .groupby(
        ["Category", "Defect Type"],
        as_index=False,
    )
    .agg(
        Images=("Filename", "size"),
        Mean_IoU=("IoU", "mean"),
        Mean_Dice=("Dice", "mean"),
        Mean_Precision=("Precision", "mean"),
        Mean_Recall=("Recall", "mean"),
    )
)

display(
    defect_type_summary_df.round(6)
)

save_table(
    defect_type_summary_df,
    "VisionGuard AI Localization Summary by Defect Type",
)

,Category,Defect Type,Images,Mean_IoU,Mean_Dice,Mean_Precision,Mean_Recall
0,bottle,broken_large,20,0.000000,0.000000,0.000000,0.000000
1,bottle,broken_small,22,0.000000,0.000000,0.000000,0.000000
2,bottle,contamination,21,0.000000,0.000000,0.000000,0.000000
3,capsule,crack,23,0.000000,0.000000,0.000000,0.000000
4,capsule,faulty_imprint,22,0.000000,0.000000,0.000000,0.000000
5,capsule,poke,21,0.000000,0.000000,0.000000,0.000000
6,capsule,scratch,23,0.000000,0.000000,0.000000,0.000000
7,capsule,squeeze,20,0.000293,0.000583,0.008333,0.000302
8,hazelnut,crack,18,0.052569,0.094832,0.560301,0.053046
9,hazelnut,cut,17,0.064678,0.106329,0.317532,0.070002


,Category,Defect Type,Images,Mean_IoU,Mean_Dice,Mean_Precision,Mean_Recall
0,bottle,broken_large,20,0.000000,0.000000,0.000000,0.000000
1,bottle,broken_small,22,0.000000,0.000000,0.000000,0.000000
2,bottle,contamination,21,0.000000,0.000000,0.000000,0.000000
3,capsule,crack,23,0.000000,0.000000,0.000000,0.000000
4,capsule,faulty_imprint,22,0.000000,0.000000,0.000000,0.000000
5,capsule,poke,21,0.000000,0.000000,0.000000,0.000000
6,capsule,scratch,23,0.000000,0.000000,0.000000,0.000000
7,capsule,squeeze,20,0.000293,0.000583,0.008333,0.000302
8,hazelnut,crack,18,0.052569,0.094832,0.560301,0.053046
9,hazelnut,cut,17,0.064678,0.106329,0.317532,0.070002


## Defect-type IoU visualization

In [53]:
fig = px.bar(
    defect_type_summary_df,
    x="Defect Type",
    y="Mean_IoU",
    color="Category",
    barmode="group",
    text="Mean_IoU",
    title="VisionGuard AI — Mean Localization IoU by Defect Type",
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{y:.3f}",
    textposition="outside",
)

fig.update_layout(
    xaxis_title="Actual Defect Type",
    yaxis_title="Mean IoU",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Mean Localization IoU by Defect Type",
)

fig.show()

## Ground-truth versus predicted coverage

In [54]:
coverage_plot_df = pixel_metrics_df.copy()

fig = px.scatter(
    coverage_plot_df,
    x="Ground Truth Coverage",
    y="Predicted Coverage",
    color="Category",
    hover_data=["Defect Type", "Filename", "IoU"],
    title="VisionGuard AI — Predicted Versus Ground Truth Defect Coverage",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Ground Truth Coverage",
    yaxis_title="Predicted Coverage",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Predicted Versus Ground Truth Defect Coverage",
)

fig.show()

## IoU versus reconstruction error

In [55]:
error_iou_df = pixel_metrics_df.merge(
    localization_df[
        [
            "Category",
            "Defect Type",
            "Filename",
            "Reconstruction MSE",
            "Anomaly Coverage",
        ]
    ],
    on=[
        "Category",
        "Defect Type",
        "Filename",
    ],
    how="left",
)

fig = px.scatter(
    error_iou_df,
    x="Reconstruction MSE",
    y="IoU",
    color="Category",
    size="Anomaly Coverage",
    hover_data=["Defect Type", "Filename"],
    title="VisionGuard AI — Reconstruction Error Versus Localization IoU",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Reconstruction MSE",
    yaxis_title="Localization IoU",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Reconstruction Error Versus Localization IoU",
)

fig.show()

## Connected component statistics

In [56]:
component_rows = []

for _, row in localization_df.iterrows():
    components = extract_components(
        row["_cleaned_mask"]
    )

    for rank, (_, component) in enumerate(
        components.iterrows(),
        start=1,
    ):
        component_rows.append({
            "Category": row["Category"],
            "Defect Type": row["Defect Type"],
            "Filename": row["Filename"],
            "Component Rank": rank,
            "Area": int(component["Area"]),
            "Width": int(component["Width"]),
            "Height": int(component["Height"]),
        })

components_df = pd.DataFrame(
    component_rows
)

display(
    display_safe(components_df).head(20)
)

if not components_df.empty:
    save_table(
        components_df,
        "VisionGuard AI Connected Component Statistics",
    )

,Category,Defect Type,Filename,Component Rank,Area,Width,Height
0,screw,good,001.png,1,12,4,3
1,screw,good,002.png,1,36,12,3
2,screw,good,002.png,2,12,4,3
3,screw,good,002.png,3,12,4,3
4,screw,good,007.png,1,100,26,20
5,screw,good,007.png,2,12,4,3
6,screw,good,008.png,1,33,5,9
7,screw,good,010.png,1,12,4,3
8,screw,good,011.png,1,30,10,3
9,screw,good,011.png,2,12,4,3


## Connected component area distribution

In [57]:
if not components_df.empty:
    fig = px.histogram(
        components_df,
        x="Area",
        color="Category",
        nbins=50,
        title="VisionGuard AI — Predicted Defect Component Area Distribution",
        template="plotly_white",
    )

    fig.update_layout(
        xaxis_title="Connected Component Area",
        yaxis_title="Component Count",
    )

    save_plotly_figure(
        fig,
        "VisionGuard AI — Predicted Defect Component Area Distribution",
    )

    fig.show()
else:
    print("No predicted connected components were found.")

## Localization confusion counts

In [58]:
pixel_confusion_df = (
    pixel_metrics_df[
        ["Category", "TP", "FP", "FN", "TN"]
    ]
    .groupby("Category", as_index=False)
    .sum()
)

display(
    pixel_confusion_df
)

save_table(
    pixel_confusion_df,
    "VisionGuard AI Pixel Localization Confusion Counts",
)

,Category,TP,FP,FN,TN
0,bottle,0,16,78691,953485
1,capsule,5,294,19839,1765718
2,hazelnut,5251,679,33481,1107469
3,metal_nut,174,99,220499,1302940
4,screw,0,208,6572,1942916


,Category,TP,FP,FN,TN
0,bottle,0,16,78691,953485
1,capsule,5,294,19839,1765718
2,hazelnut,5251,679,33481,1107469
3,metal_nut,174,99,220499,1302940
4,screw,0,208,6572,1942916


## Pixel confusion visualization

In [59]:
confusion_long_df = pixel_confusion_df.melt(
    id_vars="Category",
    value_vars=["TP", "FP", "FN", "TN"],
    var_name="Outcome",
    value_name="Pixels",
)

fig = px.bar(
    confusion_long_df,
    x="Category",
    y="Pixels",
    color="Outcome",
    barmode="stack",
    title="VisionGuard AI — Pixel-Level Localization Outcomes",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Pixel Count",
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Pixel-Level Localization Outcomes",
)

fig.show()

## Pixel ROC-AUC by category

In [60]:
def sampled_pixel_auc(category, max_pixels=250_000):
    category_rows = localization_df[
        (localization_df["Category"] == category)
        & (localization_df["Label"] == 1)
        & (localization_df["Ground Truth Available"])
    ]

    if category_rows.empty:
        return np.nan

    score_parts = []
    target_parts = []

    for _, row in category_rows.iterrows():
        target = load_mask_array(
            row["_mask_path"]
        )

        score = np.asarray(
            row["_pixel_mae"],
            dtype=np.float32,
        )

        target = target.reshape(-1).astype(np.uint8)
        score = score.reshape(-1)

        score_parts.append(score)
        target_parts.append(target)

    scores = np.concatenate(score_parts)
    targets = np.concatenate(target_parts)

    if len(scores) > max_pixels:
        rng = np.random.default_rng(SEED)
        indices = rng.choice(
            len(scores),
            size=max_pixels,
            replace=False,
        )
        scores = scores[indices]
        targets = targets[indices]

    if len(np.unique(targets)) < 2:
        return np.nan

    return float(
        roc_auc_score(
            targets,
            scores,
        )
    )

## Build pixel ROC-AUC summary

In [61]:
pixel_auc_rows = []

for category in ready_categories:
    pixel_auc_rows.append({
        "Category": category,
        "Sampled Pixel ROC-AUC": sampled_pixel_auc(category),
    })

pixel_auc_df = pd.DataFrame(
    pixel_auc_rows
)

display(
    pixel_auc_df.round(6)
)

save_table(
    pixel_auc_df,
    "VisionGuard AI Pixel ROC AUC Summary",
)

,Category,Sampled Pixel ROC-AUC
0,screw,0.873637
1,bottle,0.548468
2,capsule,0.820390
3,metal_nut,0.322834
4,hazelnut,0.734081


,Category,Sampled Pixel ROC-AUC
0,screw,0.873637
1,bottle,0.548468
2,capsule,0.820390
3,metal_nut,0.322834
4,hazelnut,0.734081


## Pixel ROC-AUC visualization

In [62]:
fig = px.bar(
    pixel_auc_df,
    x="Category",
    y="Sampled Pixel ROC-AUC",
    text="Sampled Pixel ROC-AUC",
    title="VisionGuard AI — Sampled Pixel ROC-AUC by Category",
    template="plotly_white",
)

fig.update_traces(
    texttemplate="%{y:.3f}",
    textposition="outside",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Pixel ROC-AUC",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Sampled Pixel ROC AUC by Category",
)

fig.show()

## Visualization sample selection

In [63]:
def select_localization_examples(category):
    group = localization_df[
        localization_df["Category"].eq(category)
        & localization_df["Label"].eq(1)
        & localization_df["Ground Truth Available"]
    ].copy()

    if group.empty:
        return {}

    metric_lookup = pixel_metrics_df[
        pixel_metrics_df["Category"].eq(category)
    ]

    merged = group.merge(
        metric_lookup[
            [
                "Category",
                "Defect Type",
                "Filename",
                "IoU",
                "Dice",
            ]
        ],
        on=[
            "Category",
            "Defect Type",
            "Filename",
        ],
        how="inner",
    )

    examples = {}

    if not merged.empty:
        examples["Best IoU"] = merged.sort_values(
            "IoU",
            ascending=False,
        ).iloc[0]

        examples["Worst IoU"] = merged.sort_values(
            "IoU",
            ascending=True,
        ).iloc[0]

    highest_error = group.sort_values(
        "Reconstruction MSE",
        ascending=False,
    )

    examples["Highest Error"] = highest_error.iloc[0]

    return examples


example_registry = {
    category: select_localization_examples(category)
    for category in ready_categories
}

## Rebuild a localization result for visualization

In [64]:
def result_from_row(row):
    category = row["Category"]

    result = localize_one(
        model_registry[category],
        row["_image_path"],
        category,
    )

    target = None

    if row["Ground Truth Available"] and row["_mask_path"] is not None:
        target = load_mask_array(
            row["_mask_path"]
        )

    return result, target

## Localization diagnostic visualizer

In [65]:
def plot_localization_diagnostic(category, row, label):
    result, target = result_from_row(row)

    if target is None:
        target = np.zeros_like(
            result["cleaned_mask"],
            dtype=bool,
        )

    overlay_pred = result["image"].copy()

    pred_mask = result["cleaned_mask"]
    gt_mask = target

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(17, 9),
    )

    axes[0, 0].imshow(result["image"])
    axes[0, 0].set_title("Original")

    axes[0, 1].imshow(result["reconstruction"])
    axes[0, 1].set_title("Reconstruction")

    im = axes[0, 2].imshow(
        result["pixel_mae"],
        cmap="magma",
    )
    axes[0, 2].set_title(
        f"Pixel Error Map\nThreshold={result['pixel_threshold']:.6f}"
    )
    fig.colorbar(
        im,
        ax=axes[0, 2],
        fraction=0.046,
        pad=0.04,
    )

    axes[0, 3].imshow(
        result["raw_mask"],
        cmap="gray",
    )
    axes[0, 3].set_title("Raw Threshold Mask")

    axes[1, 0].imshow(
        result["cleaned_mask"],
        cmap="gray",
    )
    axes[1, 0].set_title(
        f"Cleaned Prediction\nPixels={int(pred_mask.sum()):,}"
    )

    axes[1, 1].imshow(
        gt_mask,
        cmap="gray",
    )
    axes[1, 1].set_title(
        f"Ground Truth\nPixels={int(gt_mask.sum()):,}"
    )

    pred_rgb = overlay_pred.copy()
    pred_rgb[pred_mask] = (
        1.0,
        0.0,
        0.0,
    )

    axes[1, 2].imshow(
        pred_rgb
    )
    axes[1, 2].set_title(
        "Prediction Overlay"
    )

    axes[1, 3].imshow(
        result["image"]
    )
    axes[1, 3].contour(
        pred_mask,
        levels=[0.5],
        linewidths=1.8,
    )
    axes[1, 3].contour(
        gt_mask,
        levels=[0.5],
        linewidths=1.8,
    )
    axes[1, 3].set_title(
        "Prediction + Ground Truth Boundaries"
    )

    for ax in axes.flat:
        ax.axis("off")

    fig.suptitle(
        (
            f"VisionGuard AI — {category} — {label}\n"
            f"{row['Defect Type']} — {row['Filename']}"
        ),
        fontsize=16,
    )

    fig.tight_layout(
        rect=(0, 0, 1, 0.94)
    )

    title = (
        f"VisionGuard AI — {category} — {label} "
        "Defect Localization Diagnostic"
    )

    save_matplotlib_figure(
        fig,
        title,
    )

    return title

## Best localization examples

In [66]:
best_example_rows = []

for category, examples in example_registry.items():
    row = examples.get("Best IoU")

    if row is None:
        continue

    title = plot_localization_diagnostic(
        category,
        localization_df[
            (localization_df["Category"] == category)
            & (localization_df["Defect Type"] == row["Defect Type"])
            & (localization_df["Filename"] == row["Filename"])
        ].iloc[0],
        "Best IoU Example",
    )

    best_example_rows.append({
        "Category": category,
        "Defect Type": row["Defect Type"],
        "Filename": row["Filename"],
        "IoU": row["IoU"],
        "Dice": row["Dice"],
        "Visualization": title,
    })

best_example_df = pd.DataFrame(
    best_example_rows
)

display(
    best_example_df.round(6)
)

2026-10-02 17:13:27.428987: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-02 17:13:32.666284: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

,Category,Defect Type,Filename,IoU,Dice,Visualization
0,screw,manipulated_front,000.png,0.000000,0.000000,VisionGuard AI — screw — Best IoU Example Defe...
1,bottle,broken_large,000.png,0.000000,0.000000,VisionGuard AI — bottle — Best IoU Example Def...
2,capsule,squeeze,001.png,0.005862,0.011655,VisionGuard AI — capsule — Best IoU Example De...
3,metal_nut,flip,004.png,0.009439,0.018701,VisionGuard AI — metal_nut — Best IoU Example ...
4,hazelnut,print,008.png,0.731313,0.844807,VisionGuard AI — hazelnut — Best IoU Example D...


## Worst localization examples

In [67]:
worst_example_rows = []

for category, examples in example_registry.items():
    row = examples.get("Worst IoU")

    if row is None:
        continue

    selected = localization_df[
        (localization_df["Category"] == category)
        & (localization_df["Defect Type"] == row["Defect Type"])
        & (localization_df["Filename"] == row["Filename"])
    ]

    if selected.empty:
        continue

    title = plot_localization_diagnostic(
        category,
        selected.iloc[0],
        "Worst IoU Example",
    )

    worst_example_rows.append({
        "Category": category,
        "Defect Type": row["Defect Type"],
        "Filename": row["Filename"],
        "IoU": row["IoU"],
        "Dice": row["Dice"],
        "Visualization": title,
    })

worst_example_df = pd.DataFrame(
    worst_example_rows
)

display(
    worst_example_df.round(6)
)

2026-10-02 17:13:37.849201: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,Category,Defect Type,Filename,IoU,Dice,Visualization
0,screw,manipulated_front,000.png,0.0,0.0,VisionGuard AI — screw — Worst IoU Example Def...
1,bottle,broken_large,000.png,0.0,0.0,VisionGuard AI — bottle — Worst IoU Example De...
2,capsule,crack,000.png,0.0,0.0,VisionGuard AI — capsule — Worst IoU Example D...
3,metal_nut,bent,000.png,0.0,0.0,VisionGuard AI — metal_nut — Worst IoU Example...
4,hazelnut,crack,003.png,0.0,0.0,VisionGuard AI — hazelnut — Worst IoU Example ...


## Highest reconstruction-error defect examples

In [68]:
high_error_example_rows = []

for category, examples in example_registry.items():
    row = examples.get("Highest Error")

    if row is None:
        continue

    selected = localization_df[
        (localization_df["Category"] == category)
        & (localization_df["Defect Type"] == row["Defect Type"])
        & (localization_df["Filename"] == row["Filename"])
    ]

    if selected.empty:
        continue

    title = plot_localization_diagnostic(
        category,
        selected.iloc[0],
        "Highest Reconstruction Error",
    )

    metrics_match = pixel_metrics_df[
        (pixel_metrics_df["Category"] == category)
        & (pixel_metrics_df["Defect Type"] == row["Defect Type"])
        & (pixel_metrics_df["Filename"] == row["Filename"])
    ]

    iou = float(metrics_match.iloc[0]["IoU"]) if not metrics_match.empty else np.nan
    dice = float(metrics_match.iloc[0]["Dice"]) if not metrics_match.empty else np.nan

    high_error_example_rows.append({
        "Category": category,
        "Defect Type": row["Defect Type"],
        "Filename": row["Filename"],
        "Reconstruction MSE": row["Reconstruction MSE"],
        "IoU": iou,
        "Dice": dice,
        "Visualization": title,
    })

high_error_example_df = pd.DataFrame(
    high_error_example_rows
)

display(
    high_error_example_df.round(6)
)

2026-10-02 17:13:44.560002: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-02 17:13:51.015587: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

,Category,Defect Type,Filename,Reconstruction MSE,IoU,Dice,Visualization
0,screw,manipulated_front,020.png,0.027445,0.000000,0.000000,VisionGuard AI — screw — Highest Reconstructio...
1,bottle,broken_large,003.png,0.108397,0.000000,0.000000,VisionGuard AI — bottle — Highest Reconstructi...
2,capsule,squeeze,001.png,0.067978,0.005862,0.011655,VisionGuard AI — capsule — Highest Reconstruct...
3,metal_nut,color,010.png,0.022952,0.000000,0.000000,VisionGuard AI — metal_nut — Highest Reconstru...
4,hazelnut,print,008.png,0.010937,0.731313,0.844807,VisionGuard AI — hazelnut — Highest Reconstruc...


## Bounding-box extraction

In [69]:
def component_boxes(binary_mask):
    components = extract_components(binary_mask)

    if components.empty:
        return []

    return [
        (
            int(row["X"]),
            int(row["Y"]),
            int(row["Width"]),
            int(row["Height"]),
            int(row["Area"]),
        )
        for _, row in components.iterrows()
    ]

## Bounding-box diagnostic visualizer

In [70]:
def plot_bounding_boxes(category, row):
    result, target = result_from_row(row)

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 5),
    )

    axes[0].imshow(result["image"])
    axes[0].set_title("Original")

    axes[1].imshow(result["cleaned_mask"], cmap="gray")
    axes[1].set_title("Predicted Defect Mask")

    axes[2].imshow(result["image"])
    axes[2].set_title("Predicted Defect Regions")

    for x, y, width, height, area in component_boxes(
        result["cleaned_mask"]
    ):
        rectangle = plt.Rectangle(
            (x, y),
            width,
            height,
            fill=False,
            linewidth=2,
        )
        axes[2].add_patch(rectangle)

        axes[2].text(
            x,
            max(0, y - 3),
            f"Area={area}",
            fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.2",
                facecolor="white",
                alpha=0.75,
            ),
        )

    for ax in axes:
        ax.axis("off")

    title = (
        f"VisionGuard AI — {category} "
        "Predicted Defect Bounding Regions"
    )

    fig.suptitle(
        title,
        fontsize=15,
    )

    fig.tight_layout(
        rect=(0, 0, 1, 0.93)
    )

    save_matplotlib_figure(
        fig,
        title,
    )

    return title

## Bounding-box examples

In [71]:
bbox_rows = []

for category, examples in example_registry.items():
    row = examples.get("Best IoU")

    if row is None:
        continue

    selected = localization_df[
        (localization_df["Category"] == category)
        & (localization_df["Defect Type"] == row["Defect Type"])
        & (localization_df["Filename"] == row["Filename"])
    ]

    if selected.empty:
        continue

    title = plot_bounding_boxes(
        category,
        selected.iloc[0],
    )

    bbox_rows.append({
        "Category": category,
        "Defect Type": row["Defect Type"],
        "Filename": row["Filename"],
        "Predicted Components": int(
            selected.iloc[0]["Components"]
        ),
        "Visualization": title,
    })

bbox_example_df = pd.DataFrame(
    bbox_rows
)

display(
    bbox_example_df
)

2026-10-02 17:13:56.691112: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,Category,Defect Type,Filename,Predicted Components,Visualization
0,screw,manipulated_front,000.png,0,VisionGuard AI — screw Predicted Defect Boundi...
1,bottle,broken_large,000.png,0,VisionGuard AI — bottle Predicted Defect Bound...
2,capsule,squeeze,001.png,2,VisionGuard AI — capsule Predicted Defect Boun...
3,metal_nut,flip,004.png,1,VisionGuard AI — metal_nut Predicted Defect Bo...
4,hazelnut,print,008.png,2,VisionGuard AI — hazelnut Predicted Defect Bou...


## Per-category IoU and Dice profile

In [72]:
profile_rows = []

for category in ready_categories:
    group = pixel_metrics_df[
        pixel_metrics_df["Category"].eq(category)
    ]

    if group.empty:
        continue

    profile_rows.append({
        "Category": category,
        "P25 IoU": float(group["IoU"].quantile(0.25)),
        "Median IoU": float(group["IoU"].median()),
        "P75 IoU": float(group["IoU"].quantile(0.75)),
        "P25 Dice": float(group["Dice"].quantile(0.25)),
        "Median Dice": float(group["Dice"].median()),
        "P75 Dice": float(group["Dice"].quantile(0.75)),
    })

localization_profile_df = pd.DataFrame(
    profile_rows
)

display(
    localization_profile_df.round(6)
)

save_table(
    localization_profile_df,
    "VisionGuard AI Localization IoU Dice Profile",
)

,Category,P25 IoU,Median IoU,P75 IoU,P25 Dice,Median Dice,P75 Dice
0,screw,0.0,0.000000,0.000000,0.0,0.00000,0.000000
1,bottle,0.0,0.000000,0.000000,0.0,0.00000,0.000000
2,capsule,0.0,0.000000,0.000000,0.0,0.00000,0.000000
3,metal_nut,0.0,0.000000,0.000000,0.0,0.00000,0.000000
4,hazelnut,0.0,0.058905,0.201126,0.0,0.11123,0.334891


,Category,P25 IoU,Median IoU,P75 IoU,P25 Dice,Median Dice,P75 Dice
0,screw,0.0,0.000000,0.000000,0.0,0.00000,0.000000
1,bottle,0.0,0.000000,0.000000,0.0,0.00000,0.000000
2,capsule,0.0,0.000000,0.000000,0.0,0.00000,0.000000
3,metal_nut,0.0,0.000000,0.000000,0.0,0.00000,0.000000
4,hazelnut,0.0,0.058905,0.201126,0.0,0.11123,0.334891


## IoU percentile profile

In [73]:
profile_plot_df = localization_profile_df.melt(
    id_vars="Category",
    value_vars=[
        "P25 IoU",
        "Median IoU",
        "P75 IoU",
    ],
    var_name="Statistic",
    value_name="IoU",
)

fig = px.bar(
    profile_plot_df,
    x="Category",
    y="IoU",
    color="Statistic",
    barmode="group",
    title="VisionGuard AI — IoU Percentile Profile",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="IoU",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — IoU Percentile Profile",
)

fig.show()

## Threshold sensitivity for selected categories

In [74]:
def threshold_sensitivity(category, row, multipliers=(0.70, 0.85, 1.00, 1.15, 1.30)):
    result, target = result_from_row(row)

    records = []

    if target is None:
        return pd.DataFrame()

    base = pixel_threshold_registry[category]

    for multiplier in multipliers:
        threshold = base * multiplier

        raw = create_raw_anomaly_mask(
            result["pixel_mae"],
            threshold,
        )

        cleaned = clean_anomaly_mask(
            raw
        )

        metrics = binary_overlap_metrics(
            cleaned,
            target,
        )

        records.append({
            "Category": category,
            "Defect Type": row["Defect Type"],
            "Filename": row["Filename"],
            "Threshold Multiplier": multiplier,
            "Threshold": threshold,
            "IoU": metrics["IoU"],
            "Dice": metrics["Dice"],
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "Predicted Coverage": metrics["Predicted Coverage"],
        })

    return pd.DataFrame(records)

## Aggregate threshold sensitivity

In [75]:
sensitivity_frames = []

for category in ready_categories:
    group = localization_df[
        (localization_df["Category"] == category)
        & (localization_df["Label"] == 1)
        & (localization_df["Ground Truth Available"])
    ]

    for _, row in group.head(5).iterrows():
        frame = threshold_sensitivity(
            category,
            row,
        )

        if not frame.empty:
            sensitivity_frames.append(frame)

threshold_sensitivity_df = (
    pd.concat(
        sensitivity_frames,
        ignore_index=True,
    )
    if sensitivity_frames
    else pd.DataFrame()
)

if not threshold_sensitivity_df.empty:
    threshold_sensitivity_summary_df = (
        threshold_sensitivity_df
        .groupby(
            ["Category", "Threshold Multiplier"],
            as_index=False,
        )
        .agg(
            Mean_IoU=("IoU", "mean"),
            Mean_Dice=("Dice", "mean"),
            Mean_Precision=("Precision", "mean"),
            Mean_Recall=("Recall", "mean"),
        )
    )

    display(
        threshold_sensitivity_summary_df.round(6)
    )

    save_table(
        threshold_sensitivity_summary_df,
        "VisionGuard AI Localization Threshold Sensitivity",
    )
else:
    threshold_sensitivity_summary_df = pd.DataFrame()
    print("Threshold sensitivity could not be calculated.")

2026-10-02 17:14:01.775477: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,Category,Threshold Multiplier,Mean_IoU,Mean_Dice,Mean_Precision,Mean_Recall
0,bottle,0.70,0.073426,0.127789,0.131491,0.136271
1,bottle,0.85,0.006648,0.013078,0.137315,0.007325
2,bottle,1.00,0.000000,0.000000,0.000000,0.000000
3,bottle,1.15,0.000000,0.000000,0.000000,0.000000
4,bottle,1.30,0.000000,0.000000,0.000000,0.000000
5,capsule,0.70,0.016414,0.031238,0.018487,0.259740
6,capsule,0.85,0.010076,0.019576,0.011685,0.162139
7,capsule,1.00,0.000000,0.000000,0.000000,0.000000
8,capsule,1.15,0.000000,0.000000,0.000000,0.000000
9,capsule,1.30,0.000000,0.000000,0.000000,0.000000


## Threshold sensitivity visualization

In [76]:
if not threshold_sensitivity_summary_df.empty:
    fig = px.line(
        threshold_sensitivity_summary_df,
        x="Threshold Multiplier",
        y="Mean_IoU",
        color="Category",
        markers=True,
        title="VisionGuard AI — Localization IoU Sensitivity to Pixel Threshold",
        template="plotly_white",
    )

    fig.update_layout(
        xaxis_title="Threshold Multiplier",
        yaxis_title="Mean IoU",
        yaxis_range=[0, 1],
    )

    save_plotly_figure(
        fig,
        "VisionGuard AI — Localization IoU Sensitivity to Pixel Threshold",
    )

    fig.show()

## Final localization scorecard

In [77]:
scorecard_rows = []

for category in ready_categories:
    category_localization = pixel_metrics_df[
        pixel_metrics_df["Category"].eq(category)
    ]

    category_threshold = pixel_threshold_df[
        pixel_threshold_df["Category"].eq(category)
    ]

    if category_localization.empty:
        continue

    scorecard_rows.append({
        "Category": category,
        "Evaluated Defect Images": len(category_localization),
        "Pixel Threshold": (
            float(category_threshold.iloc[0]["Pixel Threshold"])
            if not category_threshold.empty
            else np.nan
        ),
        "Mean IoU": float(category_localization["IoU"].mean()),
        "Median IoU": float(category_localization["IoU"].median()),
        "Mean Dice": float(category_localization["Dice"].mean()),
        "Mean Precision": float(category_localization["Precision"].mean()),
        "Mean Recall": float(category_localization["Recall"].mean()),
        "Mean Pixel Accuracy": float(category_localization["Pixel Accuracy"].mean()),
        "Pixel ROC-AUC": float(
            pixel_auc_df.loc[
                pixel_auc_df["Category"].eq(category),
                "Sampled Pixel ROC-AUC",
            ].iloc[0]
        )
        if not pixel_auc_df.loc[
            pixel_auc_df["Category"].eq(category)
        ].empty
        else np.nan,
    })

localization_scorecard_df = pd.DataFrame(
    scorecard_rows
)

display(
    localization_scorecard_df.round(6)
)

save_table(
    localization_scorecard_df,
    "VisionGuard AI Final Defect Localization Scorecard",
)

,Category,Evaluated Defect Images,Pixel Threshold,Mean IoU,Median IoU,Mean Dice,Mean Precision,Mean Recall,Mean Pixel Accuracy,Pixel ROC-AUC
0,screw,119,0.543348,0.000000,0.000000,0.000000,0.000000,0.000000,0.996523,0.873637
1,bottle,63,0.567017,0.000000,0.000000,0.000000,0.000000,0.000000,0.923748,0.548468
2,capsule,109,0.580867,0.000054,0.000000,0.000107,0.001529,0.000055,0.988726,0.820390
3,metal_nut,93,0.355388,0.000238,0.000000,0.000473,0.064516,0.000238,0.855223,0.322834
4,hazelnut,70,0.178612,0.126182,0.058905,0.191049,0.539739,0.131022,0.970215,0.734081


,Category,Evaluated Defect Images,Pixel Threshold,Mean IoU,Median IoU,Mean Dice,Mean Precision,Mean Recall,Mean Pixel Accuracy,Pixel ROC-AUC
0,screw,119,0.543348,0.000000,0.000000,0.000000,0.000000,0.000000,0.996523,0.873637
1,bottle,63,0.567017,0.000000,0.000000,0.000000,0.000000,0.000000,0.923748,0.548468
2,capsule,109,0.580867,0.000054,0.000000,0.000107,0.001529,0.000055,0.988726,0.820390
3,metal_nut,93,0.355388,0.000238,0.000000,0.000473,0.064516,0.000238,0.855223,0.322834
4,hazelnut,70,0.178612,0.126182,0.058905,0.191049,0.539739,0.131022,0.970215,0.734081


## Final scorecard visualization

In [78]:
scorecard_plot_df = localization_scorecard_df.melt(
    id_vars="Category",
    value_vars=[
        "Mean IoU",
        "Mean Dice",
        "Mean Precision",
        "Mean Recall",
        "Pixel ROC-AUC",
    ],
    var_name="Metric",
    value_name="Score",
)

fig = px.bar(
    scorecard_plot_df,
    x="Category",
    y="Score",
    color="Metric",
    barmode="group",
    title="VisionGuard AI — Final Defect Localization Scorecard",
    template="plotly_white",
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Score",
    yaxis_range=[0, 1],
)

save_plotly_figure(
    fig,
    "VisionGuard AI — Final Defect Localization Scorecard",
)

fig.show()

## Export visualization inventory

In [79]:
visualization_files = sorted(
    path.name
    for path in IMAGE_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".png", ".html"}
)

visualization_inventory_df = pd.DataFrame({
    "Artifact Type": [
        "PNG" if Path(name).suffix.lower() == ".png" else "Plotly HTML"
        for name in visualization_files
    ],
    "Visualization": visualization_files,
})

display(
    visualization_inventory_df
)

print(
    "Visualization artifacts created:",
    len(visualization_inventory_df),
)

,Artifact Type,Visualization
0,Plotly HTML,Defect Distribution Across Selected Categories...
1,Plotly HTML,Defect Type Distribution in Test Data.html
2,Plotly HTML,Defective Test Image Share by Category.html
3,PNG,Defective Test Samples and Ground-Truth Masks.png
4,Plotly HTML,Final Dataset Image Counts by Category.html
...,...,...
229,Plotly HTML,test_images_by_category.html
230,Plotly HTML,train_vs_test_composition.html
231,Plotly HTML,training_and_test_image_composition.html
232,Plotly HTML,training_images_by_category.html


Visualization artifacts created: 234


## Validate title-based visualization filenames

In [80]:
filename_audit_rows = []

for name in visualization_files:
    stem = Path(name).stem

    filename_audit_rows.append({
        "Visualization": name,
        "Has Descriptive Stem": len(stem) >= 8,
        "Has Unsupported Path Separator": (
            "/" in name or "\\" in name
        ),
    })

filename_audit_df = pd.DataFrame(
    filename_audit_rows
)

display(
    filename_audit_df
)

if not filename_audit_df.empty:
    if not filename_audit_df["Has Descriptive Stem"].all():
        raise ValueError(
            "At least one visualization filename is not descriptive."
        )

    if filename_audit_df["Has Unsupported Path Separator"].any():
        raise ValueError(
            "At least one visualization filename contains a path separator."
        )

,Visualization,Has Descriptive Stem,Has Unsupported Path Separator
0,Defect Distribution Across Selected Categories...,True,False
1,Defect Type Distribution in Test Data.html,True,False
2,Defective Test Image Share by Category.html,True,False
3,Defective Test Samples and Ground-Truth Masks.png,True,False
4,Final Dataset Image Counts by Category.html,True,False
...,...,...,...
229,test_images_by_category.html,True,False
230,train_vs_test_composition.html,True,False
231,training_and_test_image_composition.html,True,False
232,training_images_by_category.html,True,False


## Final data-quality checks

In [81]:
final_checks = {
    "Requested categories configured": set(CATEGORIES) == {
        "screw",
        "bottle",
        "capsule",
        "metal_nut",
        "hazelnut",
    },
    "At least one model available": len(model_registry) > 0,
    "Test manifest nonempty": len(test_manifest_df) > 0,
    "Localization results nonempty": len(localization_df) > 0,
    "Pixel metrics available": len(pixel_metrics_df) > 0,
    "IoU values valid": pixel_metrics_df["IoU"].between(0, 1).all(),
    "Dice values valid": pixel_metrics_df["Dice"].between(0, 1).all(),
    "Precision values valid": pixel_metrics_df["Precision"].between(0, 1).all(),
    "Recall values valid": pixel_metrics_df["Recall"].between(0, 1).all(),
    "Coverage values valid": (
        pixel_metrics_df["Predicted Coverage"].between(0, 1).all()
        and pixel_metrics_df["Ground Truth Coverage"].between(0, 1).all()
    ),
}

final_quality_df = pd.DataFrame(
    [
        {
            "Check": key,
            "Passed": value,
        }
        for key, value in final_checks.items()
    ]
)

display(
    final_quality_df
)

if not final_quality_df["Passed"].all():
    raise RuntimeError(
        "One or more final localization quality checks failed."
    )

,Check,Passed
0,Requested categories configured,True
1,At least one model available,True
2,Test manifest nonempty,True
3,Localization results nonempty,True
4,Pixel metrics available,True
5,IoU values valid,True
6,Dice values valid,True
7,Precision values valid,True
8,Recall values valid,True
9,Coverage values valid,True


## Final status

In [82]:
print("=" * 72)
print("VISIONGUARD AI — DEFECT LOCALIZATION")
print("=" * 72)
print("Models available:", len(model_registry))
print("Test images localized:", len(localization_df))
print("Defective images with ground truth:", len(pixel_metrics_df))
print("Categories evaluated:", ", ".join(sorted(model_registry.keys())))
print("Pixel localization metrics: IoU, Dice, Precision, Recall, Pixel Accuracy")
print("Pixel ROC-AUC: calculated with deterministic pixel sampling")
print("Morphological cleanup: enabled")
print("Connected-component analysis: enabled")
print("Bounding-region visualization: enabled")
print("Visualization export: PNG + Plotly HTML")
print("Final quality checks:", "PASSED")
print("=" * 72)

VISIONGUARD AI — DEFECT LOCALIZATION
Models available: 5
Test images localized: 600
Defective images with ground truth: 454
Categories evaluated: bottle, capsule, hazelnut, metal_nut, screw
Pixel localization metrics: IoU, Dice, Precision, Recall, Pixel Accuracy
Pixel ROC-AUC: calculated with deterministic pixel sampling
Morphological cleanup: enabled
Connected-component analysis: enabled
Bounding-region visualization: enabled
Visualization export: PNG + Plotly HTML
Final quality checks: PASSED


## Objects retained for downstream notebooks

The following objects remain available in the runtime for later VisionGuard notebooks:

- `DATA_ROOT`
- `CATEGORIES`
- `model_registry`
- `test_manifest_df`
- `image_threshold_df`
- `pixel_threshold_df`
- `localization_df`
- `pixel_metrics_df`
- `pixel_category_summary_df`
- `defect_type_summary_df`
- `localization_scorecard_df`

The notebook intentionally keeps internal filesystem references in private columns only and removes those columns from displayed/exported tables.